# Runnable Retry Reference

Developer-facing statements defined in `langchain_core.runnables.retry`.

# `ExponentialJitterParams: TypedDict(total=False)`

`ExponentialJitterParams` stores optional values passed to Tenacity's exponential-jitter wait strategy.

All fields are optional.

## Fields

```python
initial: float # Initial retry delay
max: float # Maximum retry delay
exp_base: float # Base used for exponential backoff
jitter: float # Maximum random delay added to the exponential wait
```

---


In [1]:
import time # Import time for measuring retry delays

from langchain_core.runnables import RunnableLambda # Import RunnableLambda
from langchain_core.runnables.retry import ExponentialJitterParams, RunnableRetry # Import retry types

attempts: dict[str, int] = {"count": 0} # Store the current execution-attempt count

def unstable_operation(number: int) -> int: # Define an operation that temporarily fails
    attempts["count"] += 1 # Increase the attempt count
    print(f"Attempt {attempts['count']}") # Display the current attempt number

    if attempts["count"] < 3: # Fail during the first two attempts
        raise ValueError("Temporary failure") # Raise an exception that triggers retry

    return number * 2 # Return the successful result on the third attempt

jitter_settings: ExponentialJitterParams = { # Configure exponential retry delays
    "initial": 0.2, # Start with a short base delay
    "max": 1.0, # Limit the maximum delay to one second
    "exp_base": 2.0, # Double the exponential portion after each retry
    "jitter": 0.1, # Add up to 0.1 seconds of random delay
} # Finish defining the jitter configuration

base_runnable: RunnableLambda = RunnableLambda(unstable_operation) # Wrap the function as a Runnable

retry_runnable: RunnableRetry[int, int] = RunnableRetry( # Create the retry-enabled Runnable
    bound=base_runnable, # Set the operation that may temporarily fail
    retry_exception_types=(ValueError,), # Retry only when ValueError occurs
    wait_exponential_jitter=True, # Enable exponential backoff with jitter
    exponential_jitter_params=jitter_settings, # Supply the custom delay settings
    max_attempt_number=3, # Allow at most three total attempts
) # Finish creating the retry wrapper

start_time: float = time.perf_counter() # Record the execution start time

result: int = retry_runnable.invoke(10) # Execute the Runnable with retry behaviour

elapsed_time: float = time.perf_counter() - start_time # Calculate the total execution time

print("Final result:", result) # Display the successful result

print(f"Elapsed time: {elapsed_time:.2f} seconds") # Display the approximate retry duration

Attempt 1
Attempt 2
Attempt 3
Final result: 20
Elapsed time: 0.80 seconds



# `RunnableRetry: RunnableBindingBase[Input, Output]`

`RunnableRetry` wraps another `Runnable` and retries it when configured exception types are raised.

It is normally created through `runnable.with_retry()`, although it can also be instantiated directly.

Retry should usually be applied only to the smallest operation likely to fail temporarily, such as a network request.

## Fields

```python
bound: Runnable[Input, Output] # Underlying Runnable executed and retried
kwargs: Mapping[str, Any] # Keyword arguments supplied automatically during execution
config: RunnableConfig # Configuration merged into call-time configuration
config_factories: list[Callable[[RunnableConfig], RunnableConfig]] # Functions that generate additional configuration
custom_input_type: Any | None # Optional replacement input type
custom_output_type: Any | None # Optional replacement output type
retry_exception_types: tuple[type[BaseException], ...] = (Exception,) # Exception types that trigger another attempt
wait_exponential_jitter: bool = True # Whether retry delays use exponential backoff with jitter
exponential_jitter_params: ExponentialJitterParams | None = None # Optional exponential-jitter settings
max_attempt_number: int = 3 # Maximum total number of attempts
```

## Constructor

```python
RunnableRetry(
    *,
    bound: Runnable[Input, Output], # Underlying Runnable executed and retried
    kwargs: Mapping[str, Any] | None = None, # Keyword arguments permanently bound to the Runnable
    config: RunnableConfig | None = None, # Configuration permanently bound to the Runnable
    config_factories: list[Callable[[RunnableConfig], RunnableConfig]] | None = None, # Functions that derive additional configuration
    custom_input_type: type[Input] | BaseModel | None = None, # Optional replacement input type
    custom_output_type: type[Output] | BaseModel | None = None, # Optional replacement output type
    retry_exception_types: tuple[type[BaseException], ...] = (Exception,), # Exception types that allow retry
    wait_exponential_jitter: bool = True, # Whether to wait using exponential jitter
    exponential_jitter_params: ExponentialJitterParams | None = None, # Optional wait-strategy parameters
    max_attempt_number: int = 3, # Maximum number of total attempts
    name: str | None = None, # Optional name used for tracing and debugging
    **other_kwargs: Any, # Additional serializable model fields
) -> None # Initialize the retry wrapper
```

## Overridden Methods

### `invoke`

Synchronously executes the wrapped `Runnable` and retries it when a configured exception occurs.

Each retry attempt receives a child callback tagged with its attempt number.

The final exception is raised after all permitted attempts fail.

### `ainvoke`

Asynchronously executes the wrapped `Runnable` and retries it when a configured exception occurs.

It follows the same exception and attempt rules as `invoke()`.

### `batch`

Executes a synchronous batch and retries only the inputs that failed.

Inputs that already succeeded are not executed again during later attempts.

The final result order matches the original input order.

### `abatch`

Executes an asynchronous batch and retries only the inputs that failed.

Inputs that already succeeded are preserved and excluded from later attempts.

The final result order matches the original input order.

## Batch Behaviour

- Every input is tracked independently.
- Successful inputs are removed from later retry attempts.
- Failed inputs continue to the next attempt.
- `return_exceptions=True` returns final exceptions in the result list.
- `return_exceptions=False` raises the final failure.
- An empty input list returns an empty output list.

## Retry Behaviour

- `max_attempt_number` includes the initial attempt.
- Only exceptions listed in `retry_exception_types` trigger another attempt.
- Unlisted exception types are raised immediately.
- Exponential jitter is enabled by default.
- Setting `wait_exponential_jitter=False` disables the configured wait strategy.
- Retry callbacks are tagged as `retry:attempt:n` from the second attempt onward.
- The wrapped Runnable's bound arguments, configuration, types, and other binding behaviour are preserved.

## Streaming Limitation

`stream()`, `astream()`, `transform()`, and `atransform()` are inherited without retry-specific behaviour.

Streaming is not retried because restarting a partially consumed stream could duplicate or reorder output.

## Recommended Creation

```python
retry_runnable = runnable.with_retry(
    retry_if_exception_type=(ValueError,), # Retry only selected exception types
    wait_exponential_jitter=True, # Enable exponential backoff with jitter
    stop_after_attempt=3, # Allow at most three total attempts
    exponential_jitter_params={"initial": 1.0}, # Customize the retry delay
) # Return a RunnableRetry wrapper
```


In [ ]:
from langchain_core.runnables import RunnableLambda # Import RunnableLambda
from langchain_core.runnables.retry import RunnableRetry # Import RunnableRetry

attempts: dict[str, int] = {"count": 0} # Store the number of execution attempts

def unstable_operation(number: int) -> int: # Define an operation that temporarily fails
    attempts["count"] += 1 # Increase the attempt counter
    print("Attempt:", attempts["count"]) # Display the current attempt number

    if attempts["count"] < 3: # Fail during the first two attempts
        raise ValueError("Temporary failure") # Raise an exception that triggers retry

    return number * 2 # Return the successful result on the third attempt

base_runnable: RunnableLambda = RunnableLambda(unstable_operation) # Convert the function into a Runnable

retry_runnable: RunnableRetry[int, int] = RunnableRetry( # Create the retry wrapper directly
    bound=base_runnable, # Set the Runnable that may fail
    retry_exception_types=(ValueError,), # Retry only when ValueError is raised
    wait_exponential_jitter=False, # Disable waiting between retries for this demonstration
    max_attempt_number=3, # Allow a maximum of three total attempts
) # Finish creating RunnableRetry

result: int = retry_runnable.invoke(10) # Execute the Runnable with automatic retries

print("Final result:", result) # Display the successful result


## Developer-Facing Top-Level Statements
```python
ExponentialJitterParams # Optional exponential-jitter configuration
RunnableRetry # Runnable wrapper providing retry behaviour
```